## 1. Run configuration
 Edit all parameter used on the workflow:
 - File path
 - Patch strategy
 - VAE clustering
 - Autogluon setup
 - Stick inspection

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Local input files (do not add private/raw data to a public repository without permission).
HEADER_PATH = Path('/yourdata.hdr')
IMAGE_PATH = Path('/yourdata.img')
LABEL_CSV = Path('/yourlabel.csv') # This workflow has existing label for accuracy assesment and assist the label assignment
LABEL_INDEX_COLUMN = "Unnamed: 0"
LABEL_VALUE_COLUMN = "0"
RESULTS_ROOT = Path("results")

FEATURE_BANDS = 242
EXCLUDE_CLASS_ID = 107
MINERAL_LABELS = (101, 102, 103, 104, 105, 106)
RANDOM_SEED = 0

# Patch strategy: 'random_spatial' or 'pca_guided'
SAMPLING_STRATEGY = "pca_guided"
PATCH_SIZE = 15
N_PATCHES_TARGET = 500       # PCA representative target; valid-window filtering may reduce the final count. 500 for the 380 patches, 1000 for 770 patches and 10000 for 8270 patches
MAX_CANDIDATES = 300_000
MIN_VALID_FRACTION = 0.85
PATCH_PCA_COMPONENTS = 20
CANDIDATE_SEED = 0
SAMPLING_SEED = 0

# VAE and pixel-level clustering
VAE_LATENT_DIM = 10
VAE_HIDDEN_DIMS = (512, 256, 128)
VAE_BETA_KL = 0.5
VAE_EPOCHS = 50
VAE_BATCH_SIZE = 1024
VAE_MAX_TRAINING_SPECTRA = 200_000
VAE_EARLY_STOPPING_PATIENCE = 5
N_PIXEL_CLUSTERS = 14
N_RAW_SPECTRA_PER_CLUSTER = 5
PCA_SCORE_COMPONENTS = 10
CLUSTER_METRIC_SUBSAMPLE = 20_000

# AutoGluon model selection
MODEL_MODE = "single_model"       # 'automl' or 'single_model'
SINGLE_MODEL = "LightGBM"         # LightGBM, CatBoost, RandomForest, ExtraTrees, XGBoost, NeuralNetwork if single model is selected on the MODEL_MODE
AUTOML_PRESET = "medium_quality"  # fixed model portfolio is defined in model_tools.py
AUTOML_TIME_LIMIT_SECONDS = 3600  # one-hour AutoGluon time budget (best effort)
NUM_BAG_FOLDS = 5                 # five-fold bagging
NUM_BAG_SETS = 1
NUM_STACK_LEVELS = 1              # one stacked level (BAG_L2)
BALANCE_CLASSES = True
PREDICTION_MODEL = None            # None/'best' or exact trained model name, e.g. 'LightGBM_BAG_L1'
PREDICTION_BATCH_SIZE = 200_000
SMOKE_TEST = False
SMOKE_MAX_PIXELS = 200_000

# Stick inspection 
CORE_WIDTH_PX = None
CORE_CENTERS_PX = None
STICK_ID_TO_DISPLAY = None         # None selects the highest-disagreement stick, can also show the lowest-disagrement

## 2. Import helper from py files and load the HSI

All functions are saved in separated py files, this cell is to import them all to be used on the subsequent processes.   
The second part of the scripts is to load HSI data and normzlize it using minmax


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import importlib
import inspect
import data_tools, patch_tools, vae_tools, model_tools

# Reload local helpers so a long-running Jupyter kernel does not keep an old
# sampler signature after the .py file has been replaced.
for _module in (data_tools, patch_tools, vae_tools, model_tools):
    importlib.reload(_module)

from data_tools import load_envi_cube, load_label_map, BandMinMaxNormalizer
from patch_tools import select_patches, plot_patch_distribution
from vae_tools import (
    train_vae, collect_patch_pixels, cluster_pixels,
    compare_pca_vae_clusters, plot_random_raw_spectra,
    summarize_and_merge_clusters,
)
from model_tools import (
    fit_tabular_model, predict_full_cube, evaluate_maps,
    plot_prediction_comparison, metric_table, detect_core_sticks,
    summarize_core_sticks, plot_core_stick,
)

if "n_patches_target" not in inspect.signature(select_patches).parameters:
    raise RuntimeError(
        "The loaded patch_tools.py is outdated and does not support N_PATCHES_TARGET. "
        f"Replace it with the version packaged with this notebook, then rerun this cell. "
        f"Currently loaded: {patch_tools.__file__}"
    )
print(f"Sampler helper: {patch_tools.__file__}")

cube, wavelengths, envi_metadata = load_envi_cube(HEADER_PATH, IMAGE_PATH)
H, W, B = cube.shape
if B < FEATURE_BANDS:
    raise ValueError(f"FEATURE_BANDS={FEATURE_BANDS}, but the cube has only {B} bands.")
print(f"Cube: {H:,} rows × {W:,} columns × {B} bands")
print(f"Wavelength range: {wavelengths[0]:.1f}–{wavelengths[-1]:.1f} nm")

label_map = load_label_map(
    LABEL_CSV, (H, W), index_column=LABEL_INDEX_COLUMN, label_column=LABEL_VALUE_COLUMN
)
print(f"Class {EXCLUDE_CLASS_ID} reference pixels: {np.count_nonzero(label_map == EXCLUDE_CLASS_ID):,}")
normalizer = BandMinMaxNormalizer.fit(
    cube, FEATURE_BANDS, label_map=label_map, exclude_label=EXCLUDE_CLASS_ID
)
RUN_OUTPUT_DIR = RESULTS_ROOT / f"{SAMPLING_STRATEGY}_seed{SAMPLING_SEED}"
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
normalizer.save(RUN_OUTPUT_DIR)


## 3. Select patches and view their distribution

Run the sampling based on the pre-defined configuration in the first cell. Print and plot the patches for preview.

In [ ]:
patches, patches_raw, patch_centers, patch_valid_fraction, valid_mask_2d, sampling_metadata = select_patches(
    cube, normalizer,
    strategy=SAMPLING_STRATEGY,
    patch_size=PATCH_SIZE,
    n_patches_target=N_PATCHES_TARGET,
    max_candidates=MAX_CANDIDATES,
    min_valid_fraction=MIN_VALID_FRACTION,
    pca_components=PATCH_PCA_COMPONENTS,
    candidate_seed=CANDIDATE_SEED,
    sampling_seed=SAMPLING_SEED,
    label_map=label_map,
    exclude_label=EXCLUDE_CLASS_ID,
    feature_bands=FEATURE_BANDS,
    output_dir=RUN_OUTPUT_DIR,
)
print(f"Selected patches: {len(patches):,}")
print(f"Requested patch target: {N_PATCHES_TARGET:,}")
print("Patch sampling metadata:", sampling_metadata)
print(f"Nominal patch-window area: {sampling_metadata['nominal_window_area_fraction']:.2%} of image")
plot_patch_distribution(cube, wavelengths, patch_centers, PATCH_SIZE, selected_index=0)


## 4. Train the VAE and cluster individual pixels (step 4-6 might be replaced with other classification method)

The VAE is trained on a seeded sample of valid normalized spectra from the selected patch windows. Each valid pixel within those windows is then encoded separately using the VAE mean encoder and assigned a VAE cluster. Pixels are associated with their original image coordinates for reference-label merging and later mapping.



In [ ]:
vae, vae_encoder_mean, vae_history, n_vae_training_spectra = train_vae(
    patches,
    label_map=label_map,
    patch_centers=patch_centers,
    exclude_label=EXCLUDE_CLASS_ID,
    latent_dim=VAE_LATENT_DIM,
    hidden_dims=VAE_HIDDEN_DIMS,
    beta_kl=VAE_BETA_KL,
    epochs=VAE_EPOCHS,
    batch_size=VAE_BATCH_SIZE,
    max_training_spectra=VAE_MAX_TRAINING_SPECTRA,
    random_seed=RANDOM_SEED,
    patience=VAE_EARLY_STOPPING_PATIENCE,
)
print(f"VAE training spectra: {n_vae_training_spectra:,}")
print(f"VAE epochs completed: {len(vae_history.history.get('loss', []))}")
vae.save_weights(RUN_OUTPUT_DIR / "vae_from_patches.weights.h5")

pixel_specs, pixel_specs_raw, pixel_coords = collect_patch_pixels(
    patches, patch_centers,
    patches_raw=patches_raw,
    label_map=label_map,
    exclude_label=EXCLUDE_CLASS_ID,
    feature_bands=FEATURE_BANDS,
)
print(f"Collected {len(pixel_specs):,} valid pixel spectra from {len(patches):,} patches")
pixel_latent, pixel_cluster, pixel_kmeans = cluster_pixels(
    pixel_specs, vae_encoder_mean,
    n_clusters=N_PIXEL_CLUSTERS,
    random_seed=RANDOM_SEED,
)
display(pd.Series(pixel_cluster).value_counts().sort_index().rename("pixel_count").to_frame())
np.save(RUN_OUTPUT_DIR / "pixel_specs.npy", pixel_specs)
np.save(RUN_OUTPUT_DIR / "pixel_specs_raw.npy", pixel_specs_raw)
np.save(RUN_OUTPUT_DIR / "pixel_coords.npy", pixel_coords)
np.save(RUN_OUTPUT_DIR / "pixel_latent.npy", pixel_latent.astype(np.float32))
np.save(RUN_OUTPUT_DIR / "pixel_cluster.npy", pixel_cluster)


## 5. Inspect raw spectra and compare cluster quality

The spectra plot uses unnormalized reflectance from the selected patch pixels. The score table compares VAE latent-space clusters with PCA-plus-k-means clusters on the same normalized pixel spectra. Silhouette is computed from a seeded pixel subsample for runtime control; Davies–Bouldin uses the full representation.


In [ ]:
plot_random_raw_spectra(
    pixel_specs_raw, pixel_cluster, wavelengths[:FEATURE_BANDS],
    n_per_cluster=N_RAW_SPECTRA_PER_CLUSTER,
    random_seed=RANDOM_SEED,
)
cluster_score_table, pca_pixel_space, pixel_cluster_pca = compare_pca_vae_clusters(
    pixel_specs, pixel_latent, pixel_cluster,
    n_clusters=N_PIXEL_CLUSTERS,
    pca_components=PCA_SCORE_COMPONENTS,
    random_seed=RANDOM_SEED,
    metric_subsample=CLUSTER_METRIC_SUBSAMPLE,
)
display(cluster_score_table)
cluster_score_table.to_csv(RUN_OUTPUT_DIR / "cluster_quality_scores.csv", index=False)


## 6. Assign mineral labels to VAE clusters

Reference labels are sampled at each pixel's original image coordinate. For each VAE cluster, the dominant non-background, non-107 label determines its mineral class. The generated merge table is printed and saved. Pixels whose cluster has no valid dominant mineral label remain unassigned (`-1`).
In this notebook, automerging was done because the label from previous study was available. Should the classification using other methods or the reference labels are not available, the merging must be done manually or no merging at all.

### Notes for different label assignment workflow:  
Assign the label from other method to the merged_pixel_labels, make sure to follow the shape of the label used in this study.


In [ ]:
cluster_summary, cluster_merge_map, merged_pixel_labels, pixel_merged_map, reference_at_pixels = summarize_and_merge_clusters(
    label_map, pixel_coords, pixel_cluster,
    valid_mineral_labels=MINERAL_LABELS,
    excluded_labels=(0, EXCLUDE_CLASS_ID),
)
display(cluster_summary)
print("Auto-generated cluster-to-mineral mapping:")
for mineral, clusters in cluster_merge_map.items():
    print(f"  {mineral}: {clusters}")
print("Merged pixel-label counts:", pd.Series(merged_pixel_labels).value_counts().sort_index().to_dict())
cluster_summary.to_csv(RUN_OUTPUT_DIR / "vae_cluster_reference_summary.csv", index=False)
pixel_label_table = pd.DataFrame({
    "row": pixel_coords[:, 0], "col": pixel_coords[:, 1],
    "vae_cluster": pixel_cluster, "reference_label": reference_at_pixels,
    "merged_label": merged_pixel_labels,
})
pixel_label_table.to_csv(RUN_OUTPUT_DIR / "merged_pixel_labels.csv", index=False)
np.save(RUN_OUTPUT_DIR / "pixel_merged_map.npy", pixel_merged_map)


## 7. Train the selectable AutoGluon predictor

Training uses the normalized sampled pixel spectra with valid VAE-cluster-derived mineral labels. Configuration such multiple model, bag fold number etc must be defined in the first cell.


In [ ]:
train_mask = np.isin(merged_pixel_labels, MINERAL_LABELS)
X_train = pixel_specs[train_mask, :FEATURE_BANDS]
y_train = merged_pixel_labels[train_mask]
if not len(y_train):
    raise RuntimeError("No VAE-cluster pixels received a valid mineral label.")
print(f"Training pixels: {len(y_train):,}; classes: {np.unique(y_train)}")
display(pd.Series(y_train).value_counts().sort_index().rename("training_pixels").to_frame())

from datetime import datetime
MODEL_DIR = RUN_OUTPUT_DIR / f"autogluon_{MODEL_MODE}_{datetime.now():%Y%m%d_%H%M%S}"
predictor, model_leaderboard, feature_columns = fit_tabular_model(
    X_train, y_train,
    output_dir=MODEL_DIR,
    mode=MODEL_MODE,
    single_model=SINGLE_MODEL,
    presets=AUTOML_PRESET,
    time_limit=AUTOML_TIME_LIMIT_SECONDS,
    num_bag_folds=NUM_BAG_FOLDS,
    num_bag_sets=NUM_BAG_SETS,
    num_stack_levels=NUM_STACK_LEVELS,
    sample_weight=BALANCE_CLASSES,
    random_seed=RANDOM_SEED,
)
display(model_leaderboard)
print("AutoGluon best model:", predictor.model_best)
print("Prediction model:", PREDICTION_MODEL or "best")

prediction_map = predict_full_cube(
    predictor, cube, normalizer, feature_columns,
    batch_size=PREDICTION_BATCH_SIZE,
    smoke_test=SMOKE_TEST,
    smoke_max_pixels=SMOKE_MAX_PIXELS,
    model=PREDICTION_MODEL,
)
np.save(RUN_OUTPUT_DIR / "prediction_map.npy", prediction_map)


## 8. Compare maps and report prediction metrics

The first comparison evaluates the generated pixel-level labels at sampled coordinates. The second evaluates the prediction against the full reference label map, excluding class 107 and restricting the metrics to classes 101–106.


In [ ]:
plot_prediction_comparison(
    prediction_map, pixel_merged_map, label_map,
    mineral_labels=MINERAL_LABELS,
)
evaluation = evaluate_maps(
    prediction_map, pixel_merged_map, label_map,
    mineral_labels=MINERAL_LABELS,
    exclude_label=EXCLUDE_CLASS_ID,
)
for comparison, result in evaluation.items():
    print(f"{comparison}: n={result['n']:,}; accuracy={result['accuracy']:.4f}; "
          f"balanced accuracy={result['balanced_accuracy']:.4f}")
    display(pd.DataFrame(result["confusion_matrix"], index=MINERAL_LABELS, columns=MINERAL_LABELS))
    display(pd.DataFrame(result["report"]).T)
prediction_metrics = metric_table(evaluation)
display(prediction_metrics)
prediction_metrics.to_csv(RUN_OUTPUT_DIR / "prediction_metrics.csv", index=False)


## 9. Physical core-stick comparison

The stick detector is to inspect prediction accuracy in detail. The table ranks sticks by prediction disagreement; the default view displays the stick with the highest and lowest disagreement.


In [ ]:
stick_map, stick_centers, estimated_core_width = detect_core_sticks(
    cube, core_width_px=CORE_WIDTH_PX, core_centers_px=CORE_CENTERS_PX
)
print(f"Estimated stick width: {estimated_core_width:.1f} px")
print(f"Detected stick centres (pixels): {stick_centers.tolist()}")
stick_metrics = summarize_core_sticks(
    stick_map, label_map, prediction_map,
    mineral_labels=MINERAL_LABELS,
    exclude_label=EXCLUDE_CLASS_ID,
)
display(stick_metrics)
stick_metrics.to_csv(RUN_OUTPUT_DIR / "physical_core_stick_metrics.csv", index=False)
ranked_sticks = stick_metrics.dropna(subset=["disagreement_fraction"]).sort_values(
    "disagreement_fraction", ascending=False
)
if ranked_sticks.empty:
    raise RuntimeError("No physical core stick has valid reference/prediction pixels to compare.")

highest_stick = ranked_sticks.iloc[0]
lowest_stick = ranked_sticks.iloc[-1]
sticks_to_plot = [("Highest disagreement", highest_stick)]
if int(lowest_stick["stick_id"]) != int(highest_stick["stick_id"]):
    sticks_to_plot.append(("Lowest disagreement", lowest_stick))

for disagreement_label, stick in sticks_to_plot:
    stick_id = int(stick["stick_id"])
    print(f"{disagreement_label} stick: {stick_id} ({stick['disagreement_fraction']:.2%})")
    plot_core_stick(
        stick_id, stick_map, label_map, prediction_map,
        mineral_labels=MINERAL_LABELS,
    )